<a href="https://colab.research.google.com/github/darrickpang/Email/blob/master/Traffic_misses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install ultralytics opencv-python-headless pandas tqdm

import os
import cv2
import math
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 31.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
from google.colab import drive
drive.mount('/content/drive')

VIDEO_PATH = "/content/drive/MyDrive/video.mp4"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# COCO class ids in YOLOv8 pretrained model
COCO_PERSON = 0
COCO_BICYCLE = 1
COCO_CAR = 2
COCO_MOTORCYCLE = 3
COCO_BUS = 5
COCO_TRUCK = 7

KEEP_CLASSES = {COCO_PERSON, COCO_BICYCLE, COCO_CAR, COCO_MOTORCYCLE, COCO_BUS, COCO_TRUCK}

CLASS_NAMES = {
    COCO_PERSON: "person",
    COCO_BICYCLE: "bicycle",
    COCO_CAR: "car",
    COCO_MOTORCYCLE: "motorcycle",
    COCO_BUS: "bus",
    COCO_TRUCK: "truck",
}

def depth_proxy_from_bbox(xyxy):
    """
    xyxy = (x1, y1, x2, y2)
    Returns proxy distance: smaller => closer.
    Uses bbox height as closeness signal.
    """
    x1, y1, x2, y2 = xyxy
    h = max(1.0, float(y2 - y1))
    return 1.0 / h

def estimate_ttc(proxy_now, proxy_prev, dt):
    """
    proxy decreases when object gets closer (bbox height increases).
    If proxy_now < proxy_prev, closing is happening.
    TTC approx: proxy_now / (proxy_prev - proxy_now)/dt
    """
    if dt <= 0:
        return None

    dproxy = proxy_now - proxy_prev  # negative if closing
    closing_rate = -dproxy / dt      # positive when closing

    # If not closing or too small rate, TTC is not meaningful
    if closing_rate <= 1e-6:
        return None

    ttc = proxy_now / closing_rate
    return float(ttc)

def is_near_miss(ttc, proxy_now, ttc_thresh=2.0, proxy_close_thresh=0.0030):
    """
    Simple v1 rule:
      - Near miss if TTC < threshold OR proxy indicates very close
    proxy_close_thresh depends on video/camera. You'll tune it.
    """
    if ttc is not None and ttc < ttc_thresh:
        return True
    if proxy_now < proxy_close_thresh:
        return True
    return False

In [7]:
# Load model
model = YOLO("yolov8n.pt")  # upgrade to yolov8s.pt for better accuracy

# Tracking state: per-track history
track_history = {}  # track_id -> dict(proxy=..., t=...)

# Output settings
OUTPUT_VIDEO = "/content/near_miss_annotated.mp4"
OUTPUT_CSV   = "/content/near_miss_events.csv"

# Open video
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
nframes = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (w, h))

events = []

frame_idx = 0
pbar = tqdm(total=nframes if nframes > 0 else None)

while True:
    ok, frame = cap.read()
    if not ok:
        break

    t_sec = frame_idx / fps

    # Ultralytics tracking (ByteTrack)
    results = model.track(
        source=frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=0.35,
        iou=0.5,
        verbose=False
    )

    annotated = frame.copy()
    near_miss_flagged = False

    r0 = results[0]
    boxes = r0.boxes

    if boxes is not None and len(boxes) > 0:
        xyxy = boxes.xyxy.cpu().numpy()
        cls  = boxes.cls.cpu().numpy().astype(int)
        conf = boxes.conf.cpu().numpy()

        # Track IDs can be None for early frames; handle safely
        ids = None
        if boxes.id is not None:
            ids = boxes.id.cpu().numpy().astype(int)

        for i in range(len(xyxy)):
            c = cls[i]
            if c not in KEEP_CLASSES:
                continue

            track_id = int(ids[i]) if ids is not None else -1
            x1, y1, x2, y2 = xyxy[i]
            p_now = depth_proxy_from_bbox((x1, y1, x2, y2))

            # Compute TTC using prior proxy
            ttc = None
            if track_id != -1 and track_id in track_history:
                prev = track_history[track_id]
                dt = t_sec - prev["t"]
                ttc = estimate_ttc(p_now, prev["proxy"], dt)

            # Update history
            if track_id != -1:
                track_history[track_id] = {"proxy": p_now, "t": t_sec}

            # Near-miss decision
            hit = is_near_miss(ttc, p_now, ttc_thresh=2.0, proxy_close_thresh=0.0030)

            # Draw box
            x1i, y1i, x2i, y2i = map(int, [x1, y1, x2, y2])
            label = CLASS_NAMES.get(c, str(c))
            id_text = f"ID {track_id}" if track_id != -1 else "ID ?"

            ttc_text = "TTC: --" if ttc is None else f"TTC: {ttc:.2f}s"
            text = f"{label} | {id_text} | {ttc_text}"

            color = (0, 0, 255) if hit else (0, 255, 0)  # red if near miss else green
            cv2.rectangle(annotated, (x1i, y1i), (x2i, y2i), color, 2)

            # Text background
            (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(annotated, (x1i, max(0, y1i - th - 8)), (x1i + tw + 6, y1i), color, -1)
            cv2.putText(annotated, text, (x1i + 3, y1i - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

            if hit:
                near_miss_flagged = True
                events.append({
                    "frame": frame_idx,
                    "time_sec": t_sec,
                    "track_id": track_id,
                    "class": label,
                    "ttc_sec": ttc,
                    "proxy": p_now,
                    "conf": float(conf[i])
                })

    # Global overlay
    if near_miss_flagged:
        cv2.rectangle(annotated, (0, 0), (w, 60), (0, 0, 255), -1)
        cv2.putText(annotated, "NEAR MISS DETECTED", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2)

    writer.write(annotated)

    frame_idx += 1
    pbar.update(1)

pbar.close()
cap.release()
writer.release()

df = pd.DataFrame(events)
df.to_csv(OUTPUT_CSV, index=False)

print("Saved:", OUTPUT_VIDEO)
print("Saved:", OUTPUT_CSV)
print("Near-miss events:", len(df))
df.head(10)

  0%|          | 0/216 [00:00<?, ?it/s]

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 228ms
Prepared 1 package in 63ms
Installed 1 package in 3ms
 + lap==0.5.12

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



100%|██████████| 216/216 [00:45<00:00,  4.72it/s]


Saved: /content/near_miss_annotated.mp4
Saved: /content/near_miss_events.csv
Near-miss events: 38


,frame,time_sec,track_id,class,ttc_sec,proxy,conf
0,4,0.067883,1,car,0.868317,0.008723,0.748428
1,25,0.424266,6,car,1.199906,0.009742,0.377229
2,29,0.492148,6,car,1.363845,0.009850,0.410868
3,31,0.526089,6,car,1.883090,0.009675,0.417820
4,33,0.560031,6,car,1.355166,0.009561,0.478601
5,35,0.593972,6,car,0.950808,0.009400,0.516527
6,42,0.712766,6,car,1.750022,0.009413,0.563005
7,52,0.882473,6,car,0.577070,0.010169,0.532126
8,54,0.916414,1,car,1.966744,0.008538,0.752663
9,54,0.916414,6,car,1.125076,0.010094,0.550098
